# Konu 10 — Afiş yarın matbaada: son kontrol

Sayfa Kahve'nin "Sessiz Okuma Gecesi" afişi yarın sabah matbaaya, akşam Instagram'a gidecek. Basıldıktan sonra fark
edilen hata geri alınmaz: "yanlız" diye yazılmış bir kelime, tarihin yanında yanlış gün, hiç yazılmamış saat, uzaktan
okunmayan açık renk bir satır. Düzeltmenin tek yolu yeniden bastırmaktır.

Bugün bu son kontrolü yapan küçük bir **üretim hattı** kuruyoruz. Matbaadaki prova masası gibi: afiş masadan masaya
geçer, her masada biri yalnızca tek bir şeye bakar ve notunu yazar; sonunda bütün notlar tek bir raporda toplanır.

| Adım | Neye bakılıyor | Kim bakıyor | Neden o |
|---|---|---|---|
| 1 | Afişte hangi yazılar var | Görseli okuyan model | Python görseldeki harfleri okuyamaz |
| 2 | Yazım hatası, eksik bilgi | Metin modeli | Türkçe bilgisi gerekir |
| 3 | Tarih ile gün tutuyor mu | Python | Takvim hesabı; model bunu güvenilir bilemez |
| 4 | Yazılar okunuyor mu | Python | Renk ölçmek bir formül işi |
| 5 | Rapor | Python ve sen | Bütün notlar tek dosyada; son karar senin |

Hücreleri yukarıdan aşağı **Shift + Enter** ile sırayla çalıştır. Yalnızca Adım 1 ve 2 `anahtar.txt` ister: 1 görsel
okuma, 1 metin isteği (Bonus 1 istek daha). Bu iki adımı yeniden çalıştırmak yeni istek göndermez; nedeni Adım 1'de.

## Isınma — Python'un tek başına yakaladıkları

Önce modelsiz iki kontrol. `halkalar.py`, hattın hazır parçalarının (halkalarının) durduğu yardımcı dosya; ilk satır
içindekileri deftere alır.

**Tahmin et:** afişte "14 Kasım 2026 Cuma" yazıyor. Takvime bakmadan söyle: doğru mu? Tahminini sohbete yaz, sonra çalıştır.

In [ ]:
from halkalar import *

afisteki = gun_adi("14 Kasım 2026")
benim = gun_adi("3 Mart 2005")
print("14 Kasım 2026:", afisteki)
print("Doğum günüm:", benim)

`benim` satırına kendi doğum gününü yaz (ayın adı yazıyla, ör. "3 Mart 2005") ve hücreyi yeniden çalıştır: doğru günü
buldu mu?

Şimdi renk. İki rengin **kontrastı**, biri ötekinden ne kadar açık, onu söyleyen bir sayı: 1 ise iki renk aynıdır, 21 ise
beyaz üstüne siyahtır. Web'de okunurluk için kullanılan kurallara (WCAG) göre küçük yazı ile zemini arasında en az
**4,5** olmalı.

**Tahmin et:** afişteki hardal sarısı yazı (`#D9A441`) kâğıt beyazı zeminde (`#F4F1EA`) bu eşiği geçer mi?

In [ ]:
mavi = "#1F3A5F"
hardal = "#D9A441"
beyaz = "#F4F1EA"

mavi_oran = kontrast(mavi, beyaz)
hardal_oran = kontrast(hardal, beyaz)
print("Mavi yazı:", mavi_oran)
print("Hardal yazı:", hardal_oran)

İki kontrolü de model değil Python yaptı: biri takvim hesabı, öbürü bir formül. Bir dil modeli 14 Kasım 2026'nın hangi
gün olduğunu hesaplamaz, kelimelerden tahmin eder; bazen tutturur, bazen tutturamaz. Python her seferinde aynı hesabı
yapar. Hattın kuralı bu: **Python'un yapabildiğini Python yapar.** Yapay zekâyı yalnızca Python'un yapamadığı iki iş
için çağırıyoruz: görseldeki yazıyı okumak ve Türkçe yazımı denetlemek.

## Adım 1 — Afişi oku ve okuduğunu sakla

İlk iki satır hattın ayarı: hangi afişi denetliyoruz, notlar hangi klasöre yazılacak. Kendi afişinle çalışırken yalnızca
bu iki satırı değiştireceksin.

In [ ]:
afis = "veri/afis.jpg"
klasor = "ciktilar/sayfa_kahve"

goster(afis)

**Bir dakika bak:** bu afişte kaç sorun görüyorsun? Sohbete yaz. Sonunda hattın bulduklarıyla karşılaştıracağız.

Metni tasarım dosyasından kopyalayabilirdin, ama matbaaya giden şey dışa aktardığın görsel dosyasıdır. Müşterinin
gönderdiği bir afişte kaynak dosya zaten elinde olmaz. `afisi_oku` afişi görseli okuyan modele (`gemma-4`) gönderir: "bütün yazıları,
düzeltmeden, satır satır yaz". Okunan metin bir dosyaya kaydedilir; buna **ara çıktı** denir, prova masasının notu
gibi. Hattı yeniden çalıştırınca afiş yeniden okunmaz (kota harcanmaz), sonraki bir adım bozulursa bu iş kaybolmaz,
modelin ne okuduğunu dosyayı açıp görürsün. Hücredeki `if`, "bu dosya zaten var mı?" diye sorar.

In [ ]:
import os

yol = f"{klasor}/okunan.txt"
if os.path.exists(yol):
    print("Afiş daha önce okunmuş; dosyadan alıyorum.")
    okunan = dosyadan_oku(yol)
else:
    okunan = afisi_oku(afis)
    dosyaya_yaz(okunan, yol)
print(okunan)

**Tahmin et:** bu hücreyi bir kez daha çalıştırırsan ilk satırda ne yazar, modele yeni istek gider mi? Dene.

Sonra `ciktilar/sayfa_kahve/okunan.txt` dosyasını VS Code'da aç ve afişle karşılaştır: model "Herkez"i olduğu gibi mi
yazmış, yoksa düzeltmiş mi? Okurken düzelten bir model hatayı raporda görünmez yapar; bu yüzden ona "düzeltme" dedik.
Model bir satırı yanlış okuduysa dosyada elle düzelt, kaydet ve bu hücreyi yeniden çalıştır: hat senin düzelttiğin
metinle devam eder.

## Adım 2 — Yazım hatası ve eksik bilgi

Okunan metni metin modeline (Konu 5'teki llama) veriyoruz ve iki şey soruyoruz: Türkçe yazım hataları ve bir etkinlik
afişinde mutlaka olması gereken tarih, saat, yer bilgilerinden hangisinin eksik olduğu. Gün adını sormuyoruz; onu
Adım 3'te Python denetleyecek. Üç tırnak (`"""`) birkaç satırlık bir metin yazmayı sağlar.

In [ ]:
istek = """Aşağıdaki yazılar bir etkinlik afişinden okundu.
1) Türkçe yazım hatalarını yaz: hatalı kelime ve doğrusu.
2) Bir etkinlik afişinde tarih, saat ve yer olmalı. Hangisi eksik?
Kısa maddelerle, Türkçe cevap ver. Gün adlarını denetleme.

"""
soru = istek + okunan
print(soru)

Adım 1'deki kalıbın aynısı: cevap dosyada varsa oradan al, yoksa sor ve kaydet. Sıcaklık düşük (0.2), çünkü denetimde
yaratıcılık istemiyoruz.

In [ ]:
yol = f"{klasor}/yazim.txt"
if os.path.exists(yol):
    print("Yazım denetimi daha önce yapılmış; dosyadan alıyorum.")
    yazim = dosyadan_oku(yol)
else:
    yazim = modele_sor(soru, 0.2)
    dosyaya_yaz(yazim, yol)
print(yazim)

### Hangi öneri doğru? Son karar senin

Cevabı madde madde oku ve her öneriye "kabul" ya da "ret" de: gerçekten bir hata mı, yoksa model doğru bir şeyi mi
"düzeltmeye" kalkıyor? Bulmadığı bir şey var mı? **Senin çıktında kaç yanlış öneri var?** Sohbete yaz.

Hocanın 30 Ekim 2026'daki denemesinde modelin cevabında şu öneriler vardı: "Herkez" yerine "Herkes"; saat eksik;
"kafemizde" yerine "kafemize"; "Moda Caddesi 12" yerine "Moda Caddesi No: 12". Bunlardan ikisini kabul etmemen gerekir.
Hangileri, neden? Model her çalıştırmada başka bir cevap verebilir; ders değişmez: model önerir, afişe neyin gireceğine
tasarımcı karar verir.

### Model cevap vermezse?

İnternet kesilebilir, sunucu meşgul olabilir. `afisi_oku` ve `modele_sor` o zaman 5 saniye bekleyip yeniden sorar, en
fazla 3 kez; olmazsa "model cevap vermedi" notu döndürür ve `dosyaya_yaz` bu notu **kaydetmez**: o adımın dosyası
oluşmaz, bağlantı gelince yalnızca o adım yeniden sorulur. Görmek için sunucu adresini bilerek bozuyoruz, sonra
düzeltiyoruz (~10 saniye). `os.listdir(klasor)` klasördeki dosyaların adlarını liste olarak verir.

**Tahmin et:** son satırdaki listede `deneme.txt` olacak mı?

In [ ]:
import halkalar

halkalar.SUNUCU = "https://api.cloudflare.invalid"
bozuk = modele_sor(soru, 0.2)
dosyaya_yaz(bozuk, f"{klasor}/deneme.txt")
halkalar.SUNUCU = "https://api.cloudflare.com"

dosyalar = os.listdir(klasor)
print(bozuk)
print(dosyalar)

Klasörde `okunan.txt` ve `yazim.txt` var, `deneme.txt` yok: kopan adımın dosyası oluşmadı, öncekilerin dosyaları yerinde.
Bu hücreyi yarıda durdurursan adres bozuk kalır; o zaman menüden **Restart**'a bas ve hücreleri baştan çalıştır. Dosyası
olan adımlar yine istek göndermez.

## Adım 3 — Tarih ile gün tutuyor mu?

Bu adım baştan sona Python, anahtar istemez. `tarihi_bul` okunan metindeki tarihi, `gunu_bul` metinde yazan gün adını
bulur; `gun_adi` gerçek günü takvimden hesaplar (Isınma'daki fonksiyon).

In [ ]:
tarih = tarihi_bul(okunan)
yazan = gunu_bul(okunan)
dogru = gun_adi(tarih)

print("Afişteki tarih:", tarih)
print("Afişte yazan gün:", yazan)
print("Takvime göre:", dogru)

Şimdi karşılaştırıp rapora girecek cümleyi kuruyoruz.

In [ ]:
if yazan == dogru:
    gun_bulgusu = "Tarih ve gün tutarlı: " + tarih + ", " + dogru + "."
else:
    gun_bulgusu = "Gün yanlış: " + tarih + " bir " + dogru + "; afişte " + yazan + " yazıyor."
print(gun_bulgusu)

Bu adımda modele hiç sormadık. Sorsaydık ne olurdu? Bonus'ta deneyebilirsin.

## Adım 4 — Yazılar okunuyor mu?

`renkleri_cikar` afişteki en yaygın 5 rengi, en yaygından başlayarak verir; en yaygın olan zemindir. Sonra her rengin
zemine göre kontrastını ölçüyoruz.

**Tahmin et:** afişe bak. Kaç renk 4,5'in altında kalacak?

In [ ]:
renkler = renkleri_cikar(afis, 5)
zemin = renkler[0]
print("Zemin:", zemin)

for renk in renkler:
    oran = kontrast(renk, zemin)
    print(renk, oran)

İlk satır zeminin kendisi (kontrast 1). Ötekileri afişle eşleştir: hangisi başlığın mavisi, hangisi açık mavi kitap,
hangisi daire, hangisi "Ücretsiz çay ikramı"?

Şimdi eşiğin altında kalanları rapor için bir listeye topluyoruz. `renkler[1:]` listenin ikinci elemanından sonrası
demek: zemini atlıyoruz.

In [ ]:
renk_bulgulari = []
for renk in renkler[1:]:
    oran = kontrast(renk, zemin)
    if oran < 4.5:
        bulgu = renk + " zeminde zor okunur (kontrast " + str(oran) + ")"
        renk_bulgulari.append(bulgu)
print(renk_bulgulari)

Python ölçtü; hangisinin yazı, hangisinin süs olduğunu bilmez. Listede bir yazı rengi varsa o yazı küçük puntoda
okunmaz; bir süs rengi varsa sorun olmayabilir. Büyük başlıklarda eşik 3'tür. Bu ölçüm her rengi ana zemine göre
ölçer: mavi bandın üstündeki beyaz "SAYFA KAHVE" yazısının zemini mavidir, onu ayrıca denetlemen gerekir.

## Adım 5 — Rapor

Masaların notlarını tek bir dosyada topluyoruz. Rapor Markdown biçiminde (`#` başlık, `-` madde); `\n` yeni satır demek.

In [ ]:
rapor = "# Son kontrol: " + afis + "\n\n"
rapor = rapor + "## Yazım ve eksik bilgi (model)\n" + yazim + "\n\n"
rapor = rapor + "## Tarih ve gün (Python)\n" + gun_bulgusu + "\n\n"
rapor = rapor + "## Renk okunurluğu (Python, eşik 4.5)\n"
for bulgu in renk_bulgulari:
    rapor = rapor + "- " + bulgu + "\n"
print(rapor)

### Senin kontrolün

Hattın en kolay büyüyen yeri burası: her yeni kontrol, rapora bir bölüm daha. Örnek kontrol: afişte Instagram adresi
var mı? Bunun için model gerekmez, `"@" in okunan` yeter.

In [ ]:
if "@" in okunan:
    kendi = "Instagram adresi yazıyor."
else:
    kendi = "Instagram adresi yok."
rapor = rapor + "\n## Benim kontrolüm\n" + kendi + "\n"

Şimdi sıra sende. Aşağıdaki boş hücreye kendi kontrolünü aynı biçimde yaz ve rapora ekle. Fikirler: afişte semtin adı
(yer bilgisi) geçiyor mu; "ücretsiz" sözü geçiyor mu; ya da müşterinin sana söylediği bir kural ("ünlem kullanmayalım"
dediyse: `"!" in okunan`).

Raporu kaydet ve VS Code'da aç: `ciktilar/sayfa_kahve/rapor.md` (sağ üstteki önizleme düğmesiyle okunaklı görünür).
Başta sohbete yazdığın sorunların hepsi raporda mı? Raporda olup senin görmediğin bir şey var mı? Adım 2'de reddettiğin
önerileri dosyada sil ya da yanına "ret" yaz: müşteriye gidecek rapor senin kararlarını taşır.

In [ ]:
dosyaya_yaz(rapor, f"{klasor}/rapor.md")

### Hattı kendi afişinle çalıştır

Müşterinin bir afişini (ya da Konu 8 setinden bir gönderiyi) `.jpg` olarak `veri/` klasörüne koy, Adım 1'in ilk
hücresinde `afis` ve `klasor` satırlarını değiştir (ör. `"veri/benim_afisim.jpg"`, `"ciktilar/benim_afisim"`), sonra
**Restart** ve hücreleri baştan çalıştır. `klasor`u unutursan hat Sayfa Kahve'nin `okunan.txt`'sini bulur ve senin
afişini hiç okumaz. Gün adı yazmayan afişte Adım 3 `yok` der; hata değil.

## Bonus — Gün sorusunu modele sorsaydık?

Adım 3'te günü Python hesapladı. Aynı soruyu metin modeline soralım (1 istek, anahtar ister).

**Tahmin et:** doğru bilecek mi?

In [ ]:
soru_gun = "14 Kasım 2026 haftanın hangi günü? Yalnızca gün adını yaz."
model_cevabi = modele_sor(soru_gun, 0.2)
print("Model :", model_cevabi)
print("Python:", afisteki)

Model doğru da bilmiş olabilir. Ders yine aynı: modelin cevabını her seferinde denetlemen gerekir, Python'un hesabını
gerekmez. Yirmi afiş denetleyeceğin bir gün bu fark büyür.